# <center> Cours 2 : Complexité <center/>

## Pourquoi s'intéresser à la complexité ?

Vous développez une application de messagerie. En phase de test, avec une dizaine d'amis, tout fonctionne parfaitement : rechercher un ancien message est quasi instantané. Le jour du lancement, l'application est adoptée par 100 000 utilisateurs... et la recherche d'un message met soudain plusieurs secondes à répondre. Des utilisateurs agacés commencent à abandonner l'application.

Que s'est-il passé ? Le code n'a pas changé, mais la **taille des données** a explosé. Un algorithme qui semble rapide sur un petit jeu de test peut devenir inutilisable à grande échelle si son temps de calcul augmente trop vite avec la taille des données.

C'est précisément ce que permet d'anticiper la **théorie de la complexité** : prévoir, avant même de lancer un programme sur des données réelles, si un algorithme va « tenir la charge » ou non — sans avoir besoin de 100 000 utilisateurs pour s'en rendre compte !


De manière générale, un algorithme effectue un traitement sur des données.
S'il produit le résultat escompté, on dit que l'algorithme est *correct*.
Mais cela ne suffit pas. On veut que l'algorithme soit *efficace*, c'est-à-dire
que le *temps de calcul* du résultat soit raisonnable.

## Plusieurs chemins mènent à Rome, mais...

Il y a souvent plusieurs façons d'arriver au résultat visé, autrement dit,
plusieurs algorithmes pour résoudre un problème.

Prenons un exemple concret : une application de musique gère une **playlist qui tourne en boucle**. Quand la chanson en cours se termine, elle repart en fin de file d'attente (elle pourra être rejouée plus tard) et la chanson suivante prend l'ancienne première position. Manipuler cette playlist revient à déplacer le premier élément d'un tableau pour le mettre à la fin : c'est ce qu'on appelle un *décalage circulaire*.

**Exemple**
* playlist de départ (identifiants de 5 chansons) : `[1, 2, 3, 4, 5]`

après le *décalage circulaire*

* nouvelle playlist : `[2, 3, 4, 5, 1]`

### Première solution
1. Échanger les 1er et 2e éléments,
2. échanger les 2e et 3e,
3. échanger les 3e et 4e,
4. et ainsi de suite jusqu'au dernier élément.

In [ ]:
def decalage1(t):
    """Effectue un décalage circulaire du tableau t vers la gauche (méthode par échanges successifs). Si t est vide, elle ne modifie rien.

    :param t: le tableau à décaler, modifié sur place.
    :type t: list
    :returns: aucun (t est modifié sur place)."""
    i = 0
    while i < len(t)-1:
        tmp = t[i]
        t[i] = t[i+1]
        t[i+1] = tmp
        i += 1
    
tab = [1,2,3,4,5]
decalage1(tab)
print(tab)

In [ ]:
def test_decalage1():
    t = [1, 2, 3, 4, 5]
    decalage1(t)
    assert t == [2, 3, 4, 5, 1]
    t2 = [42]
    decalage1(t2)
    assert t2 == [42]
    print("Test de la fonction decalage1 : ok")

test_decalage1()

### Deuxième solution
1. Sauvegarder le premier élément du tableau dans une variable,
2. décaler tous les éléments d'une case vers la gauche,
3. écrire l'élément sauvegardé en dernière position.

In [ ]:
def decalage2(t):
    """Effectue un décalage circulaire du tableau t vers la gauche (méthode par sauvegarde puis décalage).

    :param t: le tableau à décaler, modifié sur place.
    :type t: list
    :returns: aucun (t est modifié sur place ; un tableau vide reste inchangé)."""
    if len(t) == 0:
        return
    tmp = t[0]
    i = 0
    while i < len(t)-1:
        t[i] = t[i+1]
        i += 1
        
    t[len(t)-1] = tmp

tab = [1,2,3,4,5]
decalage2(tab)
print(tab)

In [ ]:
def test_decalage2():
    t = [1, 2, 3, 4, 5]
    decalage2(t)
    assert t == [2, 3, 4, 5, 1]
    t2 = [42]
    decalage2(t2)
    assert t2 == [42]
    t3 = []
    decalage2(t3)
    assert t3 == []
    print("Test de la fonction decalage2 : ok")

test_decalage2()

Les deux algorithmes sont corrects : la playlist boucle correctement dans les deux cas. Mais si l'application doit gérer des playlists de plusieurs milliers de titres, et refaire ce décalage à chaque changement de chanson pour des millions d'utilisateurs, le choix de l'algorithme n'est plus neutre du tout. La question à se poser est donc : **Lequel est le plus rapide ?**

## Mesure des performances avec `time`

Pour répondre de manière expérimentale à la question précédente, on peut comparer le temps d'exécution nécessaire à chaque algorithme pour des mêmes données en entrée.

Pour cela, on crée une fonction `gen_tab_alea` qui retourne un tableau de nombres réels aléatoires compris entre 0 et 1. Cette fonction permet de fournir des données en entrée à chaque algorithme.

In [ ]:
from random import random

def gen_tab_alea(n):
    """Retourne un tableau de n nombres réels aléatoires compris entre 0 (inclus) et 1 (exclu).

    :param n: le nombre d'éléments du tableau à générer.
    :type n: int
    :returns: le tableau de n nombres aléatoires générés.
    :rtype: list"""
    tab = []
    i = 0
    while i < n: #ou while len(tab)<n
        tab.append(random())
        i += 1
    return tab

t = gen_tab_alea(10)
print(t)

In [ ]:
def test_gen_tab_alea():
    t = gen_tab_alea(20)
    assert len(t) == 20
    for x in t:
        assert 0 <= x < 1
    assert gen_tab_alea(0) == []
    print("Test de la fonction gen_tab_alea : ok")

test_gen_tab_alea()


La bibliothèque `time` fournit un chronomètre (fonction `time`) qui permet de mesurer le temps d'exécution d'un code. La fonction `time` renvoie le nombre de secondes qui se sont écoulées depuis le début de l'ère Unix (The Epoch) le 1er janvier 1970. 

Pour utiliser ce chronomètre, il faut importer cette fonction :

In [ ]:
from time import time
?time

Pour mesurer le temps d'exécution d'un code, il faut
* lancer le chrononomètre au début du programme et l'arrêter à la fin.
* calculer ensuite la différence entre les temps de départ et d'arrivée (pour obtenir la durée en sencondes).

On peut chronométrer, par exemple, le temps nécessaire pour effectuer un décalage circulaire sur un tableau de taille 10000 avec la première méthode.

In [ ]:
tab = gen_tab_alea(10000)

tic = time()       # top départ

########### Début du traitement à chronométer ###########
decalage1(tab)
############ Fin du traitement à chronométer ############

tac = time()           # arrêt du chronomètre

print(round(1000*(tac-tic),2)," ms") 
# on convertit en ms et on arrondit au centième de ms

En exécutant plusieurs fois ce programme, on constate que le temps n'est pas toujours
le même. En effet, plusieurs processus sont actifs en permanence sur la machine (applications en tâche de fond, interface graphique, réseau...) et peuvent mobiliser le processeur. Pour atténuer ce phénomène, on peut calculer la moyenne d'un certain nombre de mesures (par exemple 500) des temps d'exécution.

In [ ]:
def mesure_decalage1(tab):

    """Retourne le temps moyen (en ms, sur 500 mesures) d'un décalage circulaire de tab avec decalage1. Si tab est vide, retourne 0.0.

    :param tab: le tableau utilisé pour les mesures.
    :type tab: list
    :returns: le temps moyen d'exécution de decalage1, en millisecondes.
    :rtype: float"""
    nb_mesures = 500
    
    tic = time()       # top départ
    
    # On fait nb_messures decalages circulaires avec decalage1
    i = 0
    while i < nb_mesures:
        decalage1(tab)
        i+=1
    
    tac = time()        # arrêt du chronomètre
    
    # On fait la moyenne des temps d'exécutions (ms) 
    # des nb_mesures exécutions et on arrondit au millième
    return round(1000*(tac-tic) / nb_mesures, 3)


# Fonction pour mesurer le temps moyen pour effectuer 
# un décalage circulaire avec la deuxième méthode
def mesure_decalage2(tab):
    """Retourne le temps moyen (en ms, sur 500 mesures) d'un décalage circulaire de tab avec decalage2. Si tab est vide, retourne 0.0.

    :param tab: le tableau utilisé pour les mesures.
    :type tab: list
    :returns: le temps moyen d'exécution de decalage2, en millisecondes.
    :rtype: float"""
    nb_mesures = 500    
    tic = time()
    i = 0
    while i < nb_mesures:
        decalage2(tab)
        i+=1
    tac = time()
    return round(1000*(tac-tic) / nb_mesures, 3)


n = 10000
tab1 = gen_tab_alea(n)
tab2 = tab1.copy()     #on fait une vraie copie des valeurs de tab1
tps = mesure_decalage1(tab1)
print("Tps moyen pour un décalage circulaire sur un tableau de taille", n," avec decalage1 :", tps, "ms")
tps = mesure_decalage2(tab2)
print("Tps moyen pour un décalage circulaire sur un tableau de taille", n, " avec decalage2 :", tps, "ms")

In [ ]:
def test_mesure_decalage1():
    """Teste la structure du résultat (pas le temps exact, qui varie)."""
    t = mesure_decalage1([1, 2, 3, 4, 5])
    assert t >= 0
    print("Test de la fonction mesure_decalage1 : ok")

test_mesure_decalage1()

In [ ]:
def test_mesure_decalage2():
    """Teste la structure du résultat (pas le temps exact, qui varie)."""
    t = mesure_decalage2([1, 2, 3, 4, 5])
    assert t >= 0
    print("Test de la fonction mesure_decalage2 : ok")

test_mesure_decalage2()

En regardant les temps d'exécution, on s'aperçoit la deuxième méthode pour le décalage circulaire est plus rapide que la première méthode.

### Visualiser l'écart

On peut représenter graphiquement ce constat en mesurant les deux méthodes sur des playlists de tailles croissantes :

In [ ]:
import matplotlib.pyplot as plt

tailles = [500, 2000, 5000, 10000]
temps1 = []
temps2 = []
for n in tailles:
    tab1 = gen_tab_alea(n)
    tab2 = tab1.copy()
    temps1.append(mesure_decalage1(tab1))
    temps2.append(mesure_decalage2(tab2))

plt.plot(tailles, temps1, marker='o', label="decalage1 (échanges successifs)")
plt.plot(tailles, temps2, marker='o', label="decalage2 (sauvegarde puis décalage)")
plt.xlabel("Taille de la playlist (n)")
plt.ylabel("Temps moyen (ms)")
plt.title("Comparaison des deux méthodes de décalage circulaire")
plt.legend()
plt.show()

## Opérations élémentaires et complexité

Il est difficile de comparer expérimentalement deux algorithmes en mesurant leur temps d'exécution : il faut implémenter les deux algorithmes et faire de nombreux tests sur le même ordinateur (la vitesse du processeur, le cache, l'accès disque variant beaucoup d'un ordinateur à un autre).

La *théorie de la complexité* permet de comparer des algorithmes sans même les implémenter, en prédisant leur comportement en fonction de la taille des données en entrée. Par exemple, elle permet de répondre à : "Si le tableau en entrée est $100$ fois plus grand, l'algorithme sera-t-il $100$ fois plus long ? $10\,000$ fois plus long ?"

Elle est basée sur le calcul du nombre d'*opérations élémentaires* : affectation, addition, soustraction, comparaison de valeurs, *etc.* On suppose qu'un ordinateur met le même temps pour faire n'importe quelle opération élémentaire (petite approximation). Ainsi, connaître le nombre d'opérations élémentaires nécessaires suffit à se faire une idée du temps d'exécution.

### Nombre d'opérations élémentaires pour le décalage circulaire

#### Première méthode

On a une itération de moins que le nombre de valeurs dans le tableau. À chaque itération, on a comme opérations élémentaires :
* $4$ affectations,
* $3$ additions,
* $1$ soustraction,
* $1$ comparaison.

On a donc $9$ opérations élémentaires par itération. On a aussi une affectation au départ (`i = 0`) ainsi qu'une comparaison et une soustraction à la fin de la boucle (`i<len(t)-1`). Si le tableau a $n$ cases, alors le nombre d'opérations élémentaires est : $9(n-1)+3= 9n-6$.

#### Deuxième méthode

On a une itération de moins que le nombre de valeurs dans le tableau. À chaque itération, on a comme opérations élémentaires :
* $2$ affectations,
* $2$ additions,
* $1$ soustraction,
* $1$ comparaison.

On a donc $6$ opérations élémentaires par itération. On a aussi $3$ affectations et une soustraction en dehors de la boucle ainsi qu'une comparaison et une soustraction à la fin de la boucle (`i<len(t)-1`). Si le tableau a $n$ cases, alors le nombre d'opérations élémentaires est : $6(n-1) + 6= 6n$.

#### Comparaison

Si le tableau fait $10\,000$ cases, le premier algorithme nécessite $89\,994$ opérations alors que le deuxième nécessite $60\,000$ opérations élémentaires. Théoriquement, pour un tableau de taille $10\,000$, le deuxième algorithme est environ 1.5 fois plus rapide ($89\,994/60\,000$). On retrouve un résultat similaire expérimentalement en comparant les temps d'exécution.

#### Estimation du temps d'exécution pour un tableau de taille 100 000

Si la taille du tableau est multipliée par $10$, alors le nombre d'opérations élémentaires est aussi multiplié par $10$ et l'algorithme prendra alors $10$ fois plus de temps. On peut ainsi estimer le temps d'exécution nécessaire pour des très grandes données sans tester expérimentalement.

*On refera un calcul complet du nombre d'opérations dans l'exercice ci-dessous, sur un nouvel exemple — c'est un exercice à refaire à chaque nouvel algorithme rencontré, c'est l'entraînement qui compte plus que la formule elle-même.*

### Exercice 1 : Nombre d'opérations et complexité*

Dans un jeu vidéo, chaque ennemi vaincu rapporte `10` points au score du joueur. Le programme suivant simule un niveau comportant `n` ennemis à vaincre les uns après les autres, et calcule le score final `s` obtenu :

In [ ]:
n = 10000
s = 0
i = 0
while i < n:
    s = s + 10
    i = i + 1

**Question 1**

Quel est le nombre d'opérations élémentaires réalisées par ce programme ?

**Réponse :** Écrire votre réponse ici.




**Question 2**

Le niveau suivant comporte $10$ fois plus d'ennemis à vaincre, soit `n = 100000` (un vrai raid de boss !). Prévoir l'évolution du temps d'exécution.

**Réponse :** Écrire votre réponse ici.




**Question 3**

Dans une version améliorée du jeu, les ennemis rencontrés à une position paire (`i` pair) sont des ennemis spéciaux qui rapportent `20` points, tandis que ceux à une position impaire rapportent toujours `10` points. Mêmes questions avec le programme suivant :

In [ ]:
n = 10000
s = 0
i = 0
while i < n:
    if i%2 == 0:
        s = s + 20
    else:
        s = s + 10
    i = i + 1

**Réponse :** Écrire votre réponse ici.




## Meilleur cas, pire cas et cas moyen

En général, le nombre d'opérations dépend non seulement de la taille des données,
mais des données elles-mêmes (on dit aussi : de l'instance).

Par exemple, si l'on recherche un élément particulier dans un tableau de taille `n`, avec de la chance, l'élément est trouvé du premier coup et le programme s'arrête après une opération : c'est le **meilleur cas**.

Mais il est intéressant de prévoir la performance pour un jeu de données
défavorable ou la **complexité dans le pire des cas**. Dans l'exemple de la recherche,
le pire des cas se produit quand l'élément n'est pas trouvé ou trouvé en dernier,
ce qui oblige à parcourir le tableau entier.

On peut aussi s'intéresser au **cas moyen** : le nombre d'opérations « typique », en moyenne sur toutes les données possibles. Il est plus délicat à étudier, car il faut faire une **hypothèse sur la distribution des données** (par exemple : l'élément recherché a autant de chances d'être à chaque position). Dans ce cours, on étudiera surtout le **pire cas**, qui offre une garantie valable pour toutes les données, et on mentionnera le meilleur cas et le cas moyen lorsqu'ils apportent un éclairage utile (tri par insertion, table de hachage...).

## Complexité asymptotique

Calculer le nombre exact d'opérations élémentaires peut s'avérer difficile et n'est pas vraiment nécessaire, une estimation suffit. 

Le nombre d'opérations élémentaires peut être vu comme une fonction mathématique dépendant de la taille de l'instance. Ainsi, les complexités des méthodes de décalage sont les fonctions $f(n) = 9n - 6$ et $f(n) = 6n $. La complexité asymptotique consiste à comparer la fonction de complexité avec des fonctions de référence puissance (linéaire, quadratique, *etc*) logarithmique, exponentielle, *etc* pour des instances de grande taille (quand $n$ est grand). 

Par exemple : 
* La complexité d'un algorithme est **linéaire** si le nombre d'opérations élémentaires est environ $Cn$ où $C$ est une constante,
* La complexité d'un algorithme est **quadratique** si le nombre d'opérations élémentaires est environ $Cn^2$ où $C$ est une constante,
* La complexité d'un algorithme est **logarithmique** si le nombre d'opérations élémentaires est environ $C\log_2(n)$ où $C$ est une constante.

La complexité d'un algorithme est notée $O(g(n))$ (prononcée *grand Ô de g(n)*) , où $g(n)$ est une fonction de référence. Une complexité linéaire est notée $O(n)$, une complexité quadratique $O(n^2)$, *etc*.

### Voir la différence : linéaire contre quadratique

Le graphique suivant illustre, pour des tailles de données `n` croissantes, l'écart entre une complexité **linéaire** ($O(n)$, comme le décalage circulaire) et une complexité **quadratique** ($O(n^2)$, comme deux boucles imbriquées). Même avec des constantes du même ordre de grandeur, l'écart devient vite spectaculaire :

In [ ]:
import matplotlib.pyplot as plt

n_valeurs = list(range(0, 501, 10))
lineaire = [9 * n for n in n_valeurs]        # ~ complexité du décalage circulaire (méthode 1)
quadratique = [5 * n * n for n in n_valeurs]  # ~ complexité de deux boucles imbriquées

plt.plot(n_valeurs, lineaire, label="O(n) : 9n opérations")
plt.plot(n_valeurs, quadratique, label="O(n²) : 5n² opérations")
plt.xlabel("Taille des données (n)")
plt.ylabel("Nombre d'opérations élémentaires")
plt.title("Croissance linéaire contre croissance quadratique")
plt.legend()
plt.show()

### Exercice 1 bis : Meilleur cas, pire cas, cas moyen**

Le serveur cherche un pseudo dans un tableau de `n` pseudos non triés, en les comparant un par un au pseudo cherché, du début à la fin du tableau, jusqu'à le trouver.

**Question 1**

Combien de comparaisons sont effectuées dans le meilleur cas ? dans le pire cas ? En déduire les complexités correspondantes.


**Réponse :** Écrire votre réponse ici.


**Question 2**

On suppose que le pseudo cherché est présent dans le tableau, et qu'il a autant de chances de se trouver à chacune des `n` positions. Combien de comparaisons fait-on en moyenne ? Quelle est la complexité dans le cas moyen ?


**Réponse :** Écrire votre réponse ici.


### Exercice 2 : Boucles imbriquées et complexité**

Un tournoi en ligne organise des duels entre `n` joueurs inscrits : chaque joueur affronte, à tour de rôle, chacun des `n` joueurs (y compris lui-même, pour simplifier le programme). Le programme suivant simule tous les duels du tournoi, en ajoutant `10` points à un compteur `s` à chaque duel simulé :

* Quelle est la complexité du programme suivant ?
* Quelle est sa complexité asymptotique ?
* Le tournoi s'agrandit et passe à `7` fois plus de joueurs inscrits : prévoir l'évolution du temps d'exécution. L'organisateur peut-il garder le même serveur ?

In [ ]:
n = 100

s = 0
i = 0
while i < n:
    j = 0
    while j < n:
        s = s + 10
        j = j + 1
    i = i + 1

**Réponse :** Écrire votre réponse ici.




### Exercice 3 : Recherche d'un maximum**

On souhaite désigner le champion d'un classement, c'est-à-dire trouver le meilleur score parmi tous les scores enregistrés dans un tableau. On compare pour cela deux algorithmes de recherche du maximum.

**Question 1**

* Définir la fonction `sup` qui prend en paramètres un entier `k` et un tableau `t` et retourne `True` si `k` est supérieur ou égal à tous les éléments d'un tableau, et `False` sinon.
* Quelle est la complexité asymptotique de la fonction `sup` ?


In [ ]:
############################## 
#   Saisir votre code ici.   #
##############################




**Question 2**

- Définir la fonction `imax` prenant en paramètre un tableau `tab`. Cette fonction retournera l'indice du premier maximum, c'est-à-dire la position du premier élément du tableau qui est supérieur à tous les élements du tableau. <BR>
Pour cela, on utilisera la fonction `sup` en prenant comme valeur pour `k` les valeurs contenues dans la tableau jusqu'à trouver le premier élément supérieur à tous les autres élements du tableau. Si le tableau est vide, la fonction retourne `-1`.

- Quelle est la complexité asymptotique de la fonction `imax` ?

In [ ]:
############################## 
#   Saisir votre code ici.   #
##############################




**Réponse :** Écrire votre réponse ici.




**Question 3**

Définir la fonction `imax2` utilisant l'algorithme standard de recherche de l'indice du (premier) maximum et comparer leur complexité. Comme `imax`, elle retourne `-1` si le tableau est vide.

In [ ]:
############################## 
#   Saisir votre code ici.   #
##############################




**Réponse :** Écrire votre réponse ici.




**Question 4**

Comparer le temps d'exécution des deux algorithmes de recherche de maximum dans le pire cas lorsque le tableau est de taille $10\,000$. <BR><BR>
Un tableau, conduisant à une exécution dans le pire cas, pourra être créé à l'aide l'instruction `list(range(10000))` qui crée le tableau `[0, 1, 2, ..., 9999]`.

In [ ]:
############################## 
#   Saisir votre code ici.   #
##############################




## Récapitulatif : les complexités à connaître

Voici les grandes familles de complexité rencontrées dans ce cours, de la plus favorable à la moins favorable. Ce tableau est la référence à garder sous la main : on y reviendra à chaque chapitre.

| Complexité | Nom | Exemples rencontrés dans ce cours |
|---|---|---|
| $O(1)$ | constante | accès `tab[i]`, `len(tab)`, `tab.append(x)`, `tab.pop()` (Tableaux) ; recherche dans une table de hachage, en moyenne (Tables de hachage) |
| $O(\log n)$ | logarithmique | recherche par dichotomie (Recherche, Récursivité) |
| $O(n)$ | linéaire | parcours d'un tableau, maximum, recherche séquentielle (Recherche), insertion ou suppression à un indice, copie (Tableaux), opérations sur les chaînes (Chaînes) ; tri par comptage $O(n+k)$ |
| $O(n \log n)$ | quasi-linéaire | tri fusion (Tri) |
| $O(n^2)$ | quadratique | deux boucles imbriquées, tris par sélection, insertion (pire cas) et à bulles (Tri), concaténation `+=` en boucle (Chaînes) |
| $O(2^n)$ | exponentielle | `fibonacci` récursif naïf (Récursivité) |

Pour comprendre pourquoi ces différences comptent, voici le nombre d'opérations pour trois tailles de données (un ordinateur effectuant environ $10^8$ opérations élémentaires par seconde) :

| Complexité | n = 10 | n = 1 000 | n = 1 000 000 |
|:---|---:|---:|---:|
| O(1) | 1 | 1 | 1 |
| O(log n) | 3 | 10 | 20 |
| O(n) | 10 | 10<sup>3</sup> | 10<sup>6</sup> (≈ 0,01 s) |
| O(n log n) | 33 | 10<sup>4</sup> | 2×10<sup>7</sup> (≈ 0,2 s) |
| O(n<sup>2</sup>) | 100 | 10<sup>6</sup> | 10<sup>12</sup> (≈ 2 h 45) |
| O(2<sup>n</sup>) | 1 024 | ≈ 10<sup>301</sup> (impossible) | ≈ 10<sup>301 030</sup> (impossible) |

**Pourquoi « impossible » ?** En théorie, l'algorithme se termine et sa durée est bien définie. Mais, à 10<sup>8</sup> opérations par seconde, 10<sup>301</sup> opérations demandent environ 10<sup>293</sup> secondes, et 10<sup>301 030</sup> opérations environ 10<sup>301 022</sup> secondes. Aucun ordinateur actuel ne peut donc exécuter un tel calcul : on dit qu'il est *infaisable en pratique*. C'est pourquoi un algorithme exponentiel n'est utilisable que pour de très petites valeurs de n (quelques dizaines tout au plus).

**À retenir :** selon le meilleur, le cas moyen ou le pire cas, la complexité d'un même algorithme peut différer (tri par insertion : $O(n)$ dans le meilleur cas, $O(n^2)$ dans le pire cas). On précise donc toujours *de quel cas* on parle ; sans précision, il s'agit du **pire cas**.

*On retrouvera la notion de complexité de manière concrète à plusieurs reprises dans la suite du cours : au chapitre Recherche (recherche séquentielle en $O(n)$ contre recherche par dichotomie en $O(\log n)$), au chapitre Tri (analyse expérimentale de la complexité des algorithmes de tri) et au chapitre Table de hachage (complexité en moyenne contre complexité dans le pire des cas).*